In [1]:
# Ray Data ActorPool Underutilization -- Minimal Reproducer
# ==========================================================
# Shows that ActorPoolStrategy(initial_size=N, min_size=1) underutilizes
# the actor pool and delivers lower throughput than
# ActorPoolStrategy(min_size=N, max_size=N), even though both
# configurations have the same maximum number of actors.
#
# No external dependencies -- uses only sleep() to simulate work.
# Tested on: Ray 2.55.1

import ray
import ray.data
import time, threading, random

print(f"Ray version: {ray.__version__}")


Ray version: 2.55.1


In [2]:
NUM_ITEMS         = 100    # total items to process
ACTOR_CPUS        = 6.0    # CPUs reserved per WorkActor
NUM_ACTORS        = 9      # max actor pool size
WORK_DELAY        = 1.0    # seconds per item (simulates actor work)
UPSTREAM_DELAY    = 15     # seconds before first item is ready (> 10s debounce)
DELIVERY_INTERVAL = 0.3    # seconds between the first NUM_ACTORS items;
                           # slow delivery gives autoscaler time to fire between items


In [3]:
class SlowSource:
    """Delivers items with a long initial delay, then slowly for the first
    NUM_ACTORS items (giving the autoscaler time to fire), then bursts."""
    def __init__(self):
        self._count = 0
    def __call__(self, batch):
        if self._count == 0:
            # Simulate slow upstream I/O (e.g. S3/NFS file listing).
            # The initial delay lets the actor scale-down debounce (10s) expire.
            time.sleep(UPSTREAM_DELAY)
        if self._count < NUM_ACTORS:
            # Slow delivery for the first N items: gives the autoscaler time to
            # observe low utilization (1 busy / N actors = 0.11) and scale down.
            time.sleep(DELIVERY_INTERVAL)
        self._count += 1
        return batch

class WorkActor:
    def __call__(self, batch):
        # Jitter breaks synchronized batch completion.
        time.sleep(WORK_DELAY + random.uniform(-0.4, 0.4))
        return batch


In [4]:
ray.shutdown()
ray.init(num_cpus=64)

def _monitor(snapshots, stop_event, interval=1.0):
    # Track active actor count via CPU utilization.
    total = ray.cluster_resources().get("CPU", 64)
    t0 = time.perf_counter()
    while not stop_event.is_set():
        try:
            avail = ray.available_resources().get("CPU", 0)  # 0 = all CPUs allocated (key absent when oversubscribed)
            busy = max(0.0, total - avail)
            # Cap at actor pool CPUs: other pipeline stages also consume CPUs
            actor_busy = min(busy, NUM_ACTORS * ACTOR_CPUS)
            snapshots.append((time.perf_counter() - t0, actor_busy, round(actor_busy / ACTOR_CPUS)))
        except Exception:
            pass
        stop_event.wait(interval)

def run_pipeline(compute, label):
    snapshots, stop_event = [], threading.Event()
    t = threading.Thread(target=_monitor, args=(snapshots, stop_event), daemon=True)
    t0 = time.perf_counter()
    ds = (
        ray.data.from_items([{"id": i} for i in range(NUM_ITEMS)])
        .map_batches(SlowSource, batch_size=1,
                     compute=ActorPoolStrategy(size=1), num_cpus=0.5)
        .map_batches(WorkActor, batch_size=1, num_cpus=ACTOR_CPUS, compute=compute)
    )
    t.start()
    ds.take_all()
    stop_event.set(); t.join(timeout=3)
    elapsed = time.perf_counter() - t0
    print(f"[{label}] {elapsed:.1f}s")
    return elapsed, snapshots

from ray.data import ActorPoolStrategy

elapsed1, snaps1 = run_pipeline(
    ActorPoolStrategy(min_size=1, max_size=NUM_ACTORS, initial_size=NUM_ACTORS,
        max_tasks_in_flight_per_actor=1),
    "ActorPool(min=1, max=N, initial=N)"
)
elapsed2, snaps2 = run_pipeline(
    ActorPoolStrategy(min_size=NUM_ACTORS, max_size=NUM_ACTORS,
        max_tasks_in_flight_per_actor=1),
    "ActorPool(min=N, max=N)"
)


2026-07-07 02:23:28,335	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8269 


/raid/weijiac/Curator/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


2026-07-07 02:23:30,780	INFO logging.py:416 -- Registered dataset logger for dataset dataset_2_0


2026-07-07 02:23:30,798	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_2_0. Full logs are in /tmp/ray/session_2026-07-07_02-23-15_847817_3368973/logs/ray-data


2026-07-07 02:23:30,800	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> ActorPoolMapOperator[MapBatches(SlowSource)] -> ActorPoolMapOperator[MapBatches(WorkActor)]


...

2026-07-07 02:24:56,970	INFO logging_progress.py:231 -- MapBatches(WorkActor): 83/100


2026-07-07 02:24:56,970	INFO logging_progress.py:233 --   Tasks: 9; Actors: 9; Queued blocks: 8 (64.0B); Resources: 54.0 CPU, 80.0B object store; [0/92 objects local]


2026-07-07 02:24:56,971	INFO logging_progress.py:192 -- ============================================


2026-07-07 02:24:59,325	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_5_0 execution finished in 32.74 seconds


[ActorPool(min=N, max=N)] 32.9s


In [5]:
def print_trace(snapshots, width=60):
    peak = max((s[2] for s in snapshots), default=0) or 1
    for ts, busy, actors in snapshots:
        bar = "#" * int(actors * width / peak)
        print(f"  {ts:7.1f}s  actors~{actors:2d}  {bar}")
    avg = sum(s[2] for s in snapshots) / len(snapshots) if snapshots else 0
    print(f"  peak={peak}  avg={avg:.1f}")

print("\n=== Run 1: ActorPool(min=1, max=N, initial=N) ===")
print_trace(snaps1)
print("\n=== Run 2: ActorPool(min=N, max=N) ===")
print_trace(snaps2)
print()
print("=" * 50)
print(f"  Run 1: {elapsed1:.1f}s")
print(f"  Run 2: {elapsed2:.1f}s")
print(f"  Run 2 is {elapsed1/elapsed2:.2f}x faster")
print("=" * 50)



=== Run 1: ActorPool(min=1, max=N, initial=N) ===
      0.0s  actors~ 0  
      1.0s  actors~ 9  ############################################################
      2.0s  actors~ 9  ############################################################
      3.0s  actors~ 9  ############################################################
      4.0s  actors~ 9  ############################################################
      5.0s  actors~ 9  ############################################################
      6.0s  actors~ 9  ############################################################
      7.0s  actors~ 9  ############################################################
      8.0s  actors~ 9  ############################################################
      9.0s  actors~ 9  ############################################################
     10.0s  actors~ 9  ############################################################
     11.0s  actors~ 9  ############################################################
 

In [6]:
# ======================================================================
# OBSERVATION
# ======================================================================
#
# We ran two configurations with 9 actors (6 CPUs each) on a 64-CPU machine:
#   Run 1: ActorPoolStrategy(initial_size=9, min_size=1, max_size=9)
#   Run 2: ActorPoolStrategy(min_size=9, max_size=9)
#
# Run 1 was slower despite having the same max actor count.
#
# The actor trace shows that in Run 1, the pool cascades down during the
# initial slow-delivery window (first items arrive slowly after a 15s upstream
# delay, letting the autoscaler observe low utilization and scale down repeatedly).
# The pool stabilizes at a reduced count and never recovers to 9.
# Run 2 (min_size=9) blocks scale-down and stays at 9 throughout.
#
# We expect Run 1 to perform at least as well as Run 2.

summary = """
  Run 1  ActorPoolStrategy(initial_size=9, min_size=1, max_size=9):
    - Pool cascades down during slow delivery window; stays there (never recovers)
    - Slower throughput (fewer actors processing items)
  Run 2  ActorPoolStrategy(min_size=9, max_size=9):
    - Pool stays at 9 throughout (min_size prevents scale-down)
    - Faster throughput
  Expected: Run 1 >= Run 2 (same max actors).
  Observed: Run 1 is significantly slower.
"""
print(summary)



  Run 1  ActorPoolStrategy(initial_size=9, min_size=1, max_size=9):
    - Pool cascades down during slow delivery window; stays there (never recovers)
    - Slower throughput (fewer actors processing items)
  Run 2  ActorPoolStrategy(min_size=9, max_size=9):
    - Pool stays at 9 throughout (min_size prevents scale-down)
    - Faster throughput
  Expected: Run 1 >= Run 2 (same max actors).
  Observed: Run 1 is significantly slower.

